# team_time

> Team totals over time (post hoc, speedup plan Part C): team net harvest per complete period, the team total, the marginal gain per added agent, and reruns of Phase 8-9 cells that rebuild their original rows exactly and add the team columns.

In [ ]:
#| default_exp team_time

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math

import numpy as np

from explRL.bench_grid import PREREG, mainGrid
from explRL.bench_metrics import benchMetrics
from explRL.harness_runs import HarnessConfig, netHarvest, runHarness, worldTime
from explRL.safety_grid import attackGrid
from explRL.safety_runs import SafetyConfig, runSafety
from explRL.team_refs import ledgerArrays


def _starts(L):
    "Start time of each ledger row: the stay start where there is one, else the event start."
    return np.where(np.isfinite(L['s0']), L['s0'], L['t'])


def teamRates(L, net, period, tEnd):
    "Team net harvest per unit time in each complete period k ([k p, (k + 1) p), (k + 1) p <= tEnd)."
    start, net = _starts(L), np.asarray(net, float)
    K = int(math.floor(tEnd / period + 1e-9))
    return [float(net[(start >= k * period) & (start < (k + 1) * period)].sum()) / period for k in range(K)]


def teamColumns(L, net, period, tEnd):
    "dict(teamWelfare=team total, teamRate_0=..., ...) for the complete periods of the run."
    out = dict(teamWelfare=float(np.asarray(net, float).sum()))
    out.update({f'teamRate_{k}': r for k, r in enumerate(teamRates(L, net, period, tEnd))})
    return out


def marginalGain(rows, metric='teamWelfare', keys=('task', 'channel')):
    "{(*keys, N): (mean team total at N - at N / 2) / (N / 2)} over the sizes whose half is present."
    m = {}
    for r in rows:
        v = float(r[metric])
        if not math.isnan(v):
            m.setdefault(tuple(r[k] for k in keys) + (int(float(r['N'])),), []).append(v)
    m = {k: float(np.mean(v)) for k, v in m.items()}
    return {k: (m[k] - m[k[:-1] + (k[-1] // 2,)]) / (k[-1] / 2) for k in sorted(m)
            if k[-1] > 1 and k[-1] % 2 == 0 and k[:-1] + (k[-1] // 2,) in m}


def teamRateCell(cell):
    """Rerun one Phase 8 main-grid (`runner='harness'`) or Phase 9 (`runner='safety'`) cell with its original runner,
    build its row exactly as that runner does, and add the team columns (period = Tmove)."""
    if cell['runner'] == 'harness':
        world, hist, _, m = runHarness(HarnessConfig.fromDict(cell['cfg']), cell['N'], cell['seed'])
        row, T = benchMetrics(cell, world, hist, m, PREREG['delta']), m['T']
    elif cell['runner'] == 'safety':
        cfg = SafetyConfig.fromDict(cell['cfg'])
        world, hist, _, row = runSafety(cfg, cell['N'], cell['seed'])
        row.update(study=cell['study'], factor=cell['factor'], level=cell['level'])
        T = worldTime(cell['N'], cfg)
    else:
        raise ValueError(f"teamRateCell does not run {cell['runner']} cells")
    L = ledgerArrays(world.ledger)
    return [{**row, **teamColumns(L, netHarvest(world, L), world.Tmove, T)}]


def teamRateGrids():
    "The Phase 8 main grid and the Phase 9 attack grid (full), for the team-rate reruns (FR-C3)."
    return dict(bench=mainGrid('full'), safety=attackGrid('full'))

## Tests

In [ ]:
import numpy as np
from explRL.team_time import *
from explRL.bench_grid import mainGrid, runCell
from explRL.safety_grid import attackGrid, runSafetyCell

# hand-made ledger: stays at 1, 3 (period 0), 5 and 9 (period 1), travel at 2; run end 11 -> two complete periods of 5
L = dict(t=np.array([1., 2., 3., 5., 9.]), s0=np.array([1., np.nan, 3., 5., 9.]))
net = np.array([2., 0., 1., -1., 4.])
assert teamRates(L, net, 5.0, 11.0) == [3 / 5, 3 / 5]
c = teamColumns(L, net, 5.0, 11.0)
assert c == dict(teamWelfare=6.0, teamRate_0=0.6, teamRate_1=0.6)
assert sum(c[f'teamRate_{k}'] for k in range(2)) * 5 == net[L['s0'] < 10].sum()     # complete periods sum to the total
assert teamRates(L, net, 5.0, 9.99) == [0.6]                                      # incomplete periods are left out
# a travel row's start time is its event time; a stay row's is its stay start
L2 = dict(t=np.array([4.0, 4.0]), s0=np.array([np.nan, 6.0]))
assert teamRates(L2, np.array([1.0, 2.0]), 5.0, 10.0) == [0.2, 0.4]

# marginal gain per added agent
rows = [dict(task='board', channel='none', N=N, seed=s, teamWelfare=v) for N, v in ((1, 10), (2, 16), (4, 20)) for s in (0, 1)]
assert marginalGain(rows) == {('board', 'none', 2): 6.0, ('board', 'none', 4): 2.0}

# reruns keep the original rows exactly and add the team columns
for cell, run in ((mainGrid('quick')[0], runCell), (attackGrid('quick')[1], runSafetyCell)):
    new, old = teamRateCell(cell)[0], run(cell)[0]
    assert {k: new[k] for k in old} == old or all(new[k] == old[k] or (new[k] != new[k] and old[k] != old[k]) for k in old)
    assert 'teamRate_0' in new and abs(new['teamWelfare'] - (old['welfare'] if 'welfare' in old else new['teamWelfare'])) < 1e-9
g = teamRateGrids()
assert len(g['bench']) == len(mainGrid('full')) and len(g['safety']) == len(attackGrid('full'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()